# MeMo HF
Version integrated with Transformer Libraries (Version 0.4)

In [1]:
import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation, EvaluationUpdateNew

In [2]:
d,h,l = 2048, 4, 2
chunk_length = h**(l+1)

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          max_length=max_length, model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id
del tokenizer.added_tokens_encoder['                        ']
del tokenizer.added_tokens_decoder[50254]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Setting pad token and pad token id = <|endoftext|>, 0


In [3]:
with open("testo_di_prova.txt") as my_first_text_f:
    my_first_text = my_first_text_f.read()

token_ids = tokenizer.get_text_batch_encoding([my_first_text])#, return_tensors='pt')
print(token_ids) # return max len + 1 

Token indices sequence length is longer than the specified maximum sequence length for this model (675 > 65). Running this sequence through the model will result in indexing errors


{'input_ids': tensor([[    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
         38577, 17622,  1073, 48505,   372,     8,  9718,    74,   843,   936,
          4164, 43876, 41380,   258,   367,   727, 20110,  5022, 26218,   313,
         15723,   445,  2721,    13],
        [ 3414,   358,  3381, 15410,    26, 25404, 14507,  1628,  9776,  1266,
            74,    13,   337,  7974, 11703,   639, 39337,  1638,  1540,    10,
         24449, 20335, 48019,   440,  2314,  4173,   299,  8913,  2942,   250,
           352,  6770,    80,    13,  2248,    80,   861,   410,   372, 32924,
          1073, 33813,   445,  2721,   299,  2248,    80,  1484, 19216,  1073,
           659,  4611,  1073,   391,   300,   466,  5711, 14804,  1431,   304,


In [4]:
memo_input = tokenizer.get_text_batch_encoding([my_first_text, my_first_text[0:10]])
memo_input.keys(), memo_input['input_ids'].shape

(dict_keys(['input_ids', 'labels']), torch.Size([12, 64]))

In [5]:
for i in range(3):
    print(tokenizer.decode(memo_input['input_ids'][i]))
    print(tokenizer.decode(memo_input['labels'][i]))
    print()

<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>Cosimo di Giovanni de' Medici detto il Vecchio o Pater patriÃ¦ (Firenze,
<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|en

Check memorization on single layer

In [6]:
memo_input = tokenizer.get_text_batch_encoding([my_first_text, my_first_text[10:30]])

memo_input['input_ids'].shape

torch.Size([12, 64])

In [7]:
from MeMoHF.modelling_memo_embedding import MeMoEmbedding

In [8]:
d,h,l = 1024, 4, 3

In [9]:
embedding = MeMoEmbedding(
    num_embeddings=tokenizer.vocab_size+1,# tokenizer.vocab_size,
    embedding_dim=d,
    padding_idx=tokenizer.pad_token_id,
    padding_seq_idx=tokenizer.vocab_size, 
    _freeze=True
)

MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0


In [10]:
input_embeddings = embedding.encode(memo_input['input_ids'])
output_symbols = embedding.encode(memo_input['labels'])

input_embeddings.shape, output_symbols.shape

(torch.Size([12, 64, 1024]), torch.Size([12, 64, 1024]))

In [11]:
input_tokens_ids = tokenizer(['Test', 'Un altro Test'])['input_ids']
print(input_tokens_ids)

input_embeddings = embedding.forward(input_tokens_ids)
print(input_embeddings)
print(embedding.decode(input_embeddings)[0])

tensor([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0, 5089],
        [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0, 2447, 6945,  287, 6004]])
tensor([[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  

In [12]:
from MeMoHF.modelling_memo_layer import MeMoLayer

In [13]:
layer = MeMoLayer(d, h)
layer

MeMoLayer(
  (W_v_single_head): ProjectionTokens(in_features=1024, out_features=256)
  (Prj): ProjectionSequence((trasposed wrt saved one) in_features=4096, out_features=1024)
  (CMM_OUT): CorrelationMatrixMemory(in_features=1024, out_features=1024)
)

Memo: Initializing the Tokenizer and the model

In [14]:
# Meta Parameters : 
#    d - inner dimension
#    h - number of heads
#    l - number of layers
d,h,l = 2048, 4, 2
chunk_length = h**(l+1)

# Initializing a standard Tokenizer
# TODO unclear what is the difference between max_lenght and model_max_lenght
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          max_length=max_length, model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

## TODO this is for the new added padding token, the tokenizer is to be modified to know that padding
del tokenizer.added_tokens_encoder['                        ']
del tokenizer.added_tokens_decoder[50254]


# Intializing Memo Configuration
config = MeMoConfig(
    vocab_size=tokenizer.vocab_size+1, #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,          
    padding_seq_idx=tokenizer.vocab_size, 
)

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Setting pad token and pad token id = <|endoftext|>, 0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0


Reading the two texts

In [15]:
with open("testo_di_prova.txt") as my_first_text_f:
    my_first_text = my_first_text_f.read()
with open("testo_di_prova2.txt") as my_first_text_f:
    my_second_text = my_first_text_f.read()



In [16]:
# batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=[my_first_text])
# outs = model.forward_with_loss_parallelized(
#     batch_inputs=batch_inputs,
# )
# print(outs['loss'])

In [17]:
# memo_input_test = tokenizer.get_text_batch_encoding([my_first_text])
# model.memorize_text(memo_input_test)

In [18]:
# batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=my_first_text)
# outs = model.forward_with_loss(
#     batch_inputs=batch_inputs,
# )
# print(outs['loss'])

Memorizing the first text and evaluating if it is memorized

In [19]:
memo_input_1 = tokenizer.get_text_batch_encoding([my_first_text]*8)  # Writing the same doc 8 times to stress the memorization with batch
memo_input_2 = tokenizer.get_text_batch_encoding([my_second_text]*8) # Writing the same doc 8 times to stress the memorization with batch

model.memorize_text(memo_input_1)

Token indices sequence length is longer than the specified maximum sequence length for this model (675 > 65). Running this sequence through the model will result in indexing errors


In [20]:
e = EvaluationUpdateNew()

e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Starting point : 8


100%|██████████| 55/55 [00:58<00:00,  1.06s/it]


Starting point : 8


100%|██████████| 55/55 [02:01<00:00,  2.21s/it]

Memorization level of first text  :  tensor(0.9983)
Memorization level of second text :  tensor(0.0288)


Memorizing the second text and checking if it affected the memorization of the first text

In [21]:
model.memorize_text(memo_input_2)

In [22]:
e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=0)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=0)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Starting point : 0


100%|██████████| 63/63 [01:08<00:00,  1.10s/it]


Starting point : 0


100%|██████████| 63/63 [02:22<00:00,  2.27s/it]

Memorization level of first text  :  tensor(0.9709)
Memorization level of second text :  tensor(0.9654)


In [23]:
#### raise error
# batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=[my_first_text]*8)
# with torch.no_grad():
#     model.eval()
#     outs = model.forward_with_loss_parallelized(
#         batch_inputs=batch_inputs
#     )
#     model.train()
# loss = outs['loss']
# del outs
# print(loss)

Forgetting one of the documents

In [24]:
model.forget_text(memo_input_2)

Checking the effect on the two texts

In [ ]:
############ TODO to check why the forget is not working in this version
e1 = e.check_pretokenized(model, tokenizer, memo_input_1['input_ids'], starting_point=8)
e2 = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'], starting_point=8)

print("Memorization level of first text  : ", e1) 
print("Memorization level of second text : ", e2) 

Starting point : 8


100%|██████████| 55/55 [01:00<00:00,  1.11s/it]


Starting point : 8


100%|██████████| 55/55 [01:56<00:00,  2.12s/it]

Memorization level of first text  :  tensor(0.9965)
Memorization level of second text :  tensor(0.9973)


In [26]:
memo_input_2['input_ids']

tensor([[    0,     0,     0,  ...,   660,   400,   311],
        [  275, 29505,   801,  ..., 42722, 26932, 33083],
        [  891,  7361,   503,  ..., 20938,  5940, 24041],
        ...,
        [   83,  2683,    74,  ...,   936,  1327,  4172],
        [ 5022, 23215,   247,  ...,    74, 30975,  6727],
        [ 1161, 43538,   826,  ...,  6193,    77,  2610]])

In [27]:
pred = model.retrieve(memo_input_2['input_ids'])
pred.logits.argmax(dim=-1).squeeze(dim=-1) == memo_input_2['labels'][..., -1]

tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, Tr

In [28]:
model

MeMoForCausalLM(
  (memo): MeMo(
    (encoder): MeMoEmbedding(50256, 2048, padding_idx=0)
    (output_encoder): MeMoEmbedding(50256, 2048, padding_idx=0)
    (layers): MeMoLayers(
      (0): MeMoLayer(
        (W_v_single_head): ProjectionTokens(in_features=2048, out_features=512)
        (Prj): ProjectionSequence((trasposed wrt saved one) in_features=8192, out_features=2048)
        (CMM_OUT): CorrelationMatrixMemory(in_features=2048, out_features=2048)
      )
      (1): MeMoLayer(
        (W_v_single_head): ProjectionTokens(in_features=2048, out_features=512)
        (Prj): ProjectionSequence((trasposed wrt saved one) in_features=8192, out_features=2048)
        (CMM): CorrelationMatrixMemory(in_features=2048, out_features=2048)
        (CMM_OUT): CorrelationMatrixMemory(in_features=2048, out_features=2048)
      )
    )
  )
  (lm_head): MeMoEmbedding(50256, 2048, padding_idx=0)
)

In [ ]:
############ not tested from here

In [ ]:
with open("testo_di_prova2.txt") as my_first_text_f:
    my_second_text = my_first_text_f.read()


In [ ]:
torch.cuda.is_available()

In [ ]:
config = MeMoConfig(
    vocab_size=tokenizer.vocab_size+1, #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,          
    padding_seq_idx=tokenizer.vocab_size, 
)

# Initializing the Memo Model from the configuration
model = MeMoForCausalLM(config)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')
# print("CMM pre learning")
# display(model.memo.layers[0].CMM.weight)


bs = 8
for b in range(bs):
    print(f"memorizing the same text iteration = {b}")
    memo_input = tokenizer.get_text_batch_encoding(my_first_text)
    model.memorize_text(memo_input)

# Prj = model.memo.layers[0].Prj.weight.detach().cpu()
# CMM = model.memo.layers[0].CMM.weight.detach().cpu()

# display(Prj.T @ Prj)
# display(CMM)

e = Evaluation() #UpdateNew()
out = e.check_pretokenized(model, tokenizer, memo_input['input_ids'])
print("Degree of memorization after memorizing 1: %f ", out)


memo_input_2 = tokenizer.get_text_batch_encoding(my_second_text) 
model.memorize_text(memo_input_2)
out = e.check_pretokenized(model, tokenizer, memo_input['input_ids'])
print("Degree of memorization of test 1 after memorizing 2: %f ", out)
out = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'])
print("Degree of memorization of test 2 after memorizing 2: %f ", out)


for b in range(bs):
    print(f"forgetting the same text iteration = {b}")
    memo_input = tokenizer.get_text_batch_encoding(my_first_text)
    model.forget_text(memo_input)

out = e.check_pretokenized(model, tokenizer, memo_input['input_ids'])
print("Degree of memorization of test 1 after forgetting 1: %f ", out)
out = e.check_pretokenized(model, tokenizer, memo_input_2['input_ids'])
print("Degree of memorization of test 2 after after forgetting 1: %f ", out)

In [ ]:
model.save_pretrained('MemoExp')
tokenizer.save_pretrained('MemoExp')

In [ ]:
model

In [ ]:
# from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
model1 = MeMoForCausalLM.from_pretrained("MemoExp", device_map="auto")
model = model1

In [ ]:
model.device

In [ ]:
model.config

In [ ]:
# code for testing generation capabilities of MeMo
model.eval()

memo_input_test = tokenizer.get_text_batch_encoding(my_first_text)
print(f'text={my_first_text}\nmemo_input={memo_input_test}')

if torch.cuda.is_available():
    generated_text = model.generate(inputs=memo_input_test['input_ids'].to('cuda'), max_new_tokens=20)
else:
    generated_text = model.generate(inputs=memo_input_test['input_ids'], max_new_tokens=20)

generated_text.shape
final_text = tokenizer.decode(generated_text[0], skip_special_token=True)
next_sequence = tokenizer.decode(generated_text[0][memo_input_test['input_ids'].shape[1]:], skip_special_tokens=True)
print(final_text)
print(next_sequence)

In [ ]:
batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=my_first_text)
outs = model.forward_with_loss(
    batch_inputs=batch_inputs,
)

In [ ]:
print(outs['loss'])